# 02 — Isodepth vs pseudotime, and cross-species coordinate transfer

**Claims under test**
1. The spatial coordinate and the trajectory coordinate agree *within each tissue section*.
2. A coordinate transferred across species through the SAMap graph matches the target species' native
   coordinate — and how that degrades by mouse timepoint.

**Direction labels matter here and I got them wrong once.** In `coordinate_transfer.json`:
`iso_to_mouse` = human isodepth transferred **onto mouse**, scored vs **mouse native pseudotime**
(cross-scalar); `dpt_to_human` = mouse pseudotime onto human, scored vs human isodepth. Those are
*not* the same quantity as transferred-isodepth-vs-native-isodepth.

In [ ]:
# --- paths & helpers (edit ROOT if you move things) ---------------------------
import os, json, glob, pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

X    = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final")
REV  = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/decidualization/revisions")
NET  = pathlib.Path("/n/netscratch/eddy_lab/Everyone/nhilgert")
GP   = REV/"crossspecies_radial_gp"
DELIV= X/"data/deliverables"
ISO  = X/"results/isodepth"
MIS  = X/"results/mi_stratified"
DATA = pathlib.Path("data")     # staged artifacts, see README
ORTH = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/human_ivd_repro/orthogroups/ensembl_human_mouse_components.tsv")

HS_DELIV = DELIV/"2026_08_28_Cevrim_Human_Only_Fib_Dec_Cells.h5ad"
MM_DELIV = DELIV/"2026_08_28_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad"   # NOTE: "XMens", not "Mouse"

_checks = []
def verify(label, got, expected, tol=0.02, note=""):
    """Recompute vs recorded. Prints PASS/MISMATCH and accumulates a summary."""
    try:
        ok = (abs(float(got) - float(expected)) <= tol * max(1.0, abs(float(expected)))
              if expected is not None else None)
    except (TypeError, ValueError):
        ok = (got == expected)
    tag = "PASS " if ok else ("MISMATCH" if ok is False else "n/a  ")
    _checks.append((label, got, expected, tag))
    print(f"[{tag}] {label}: recomputed={got}  recorded={expected}" + (f"   ({note})" if note else ""))
    return ok

def check_summary():
    d = pd.DataFrame(_checks, columns=["claim","recomputed","recorded","status"])
    n_bad = int((d.status.str.strip()=="MISMATCH").sum())
    print(f"\n{len(d)} checks | {n_bad} MISMATCH")
    return d

def ortholog_1to1():
    o = pd.read_csv(ORTH, sep="\t")
    o = o[o.homology_type=="ortholog_one2one"]
    piv = o.pivot_table(index="component_id", columns="species", values="gene_symbol",
                        aggfunc=lambda s: s.iloc[0] if len(s)==1 else np.nan).dropna()
    return piv[["human","mouse"]].drop_duplicates()

print("data dir:", DATA.resolve(), "| files:", len(list(DATA.glob("*"))) if DATA.exists() else "MISSING")
print("env ok |", "deliverables:", HS_DELIV.exists(), MM_DELIV.exists(), "| isodepth dir:", ISO.exists())


## 2.1 Within-species: isodepth vs pseudotime, pooled and per tissue

Recorded: human 0.671 overall (menstrual 0.795, secretory 0.551); mouse 0.690 overall (per-timepoint 0.606–0.840).

In [ ]:
import scanpy as sc
from scipy.stats import spearmanr
out = {}
for sp, f, grp in (("human","human_fibdec_isodepth.h5ad","phase"), ("mouse","mouse_fibdec_isodepth.h5ad","timepoint")):
    a = sc.read_h5ad(ISO/f, backed="r"); o = a.obs
    ptcol = next((c for c in ("dpt_pseudotime","palantir_pt") if c in o.columns), None)
    iso, pt = o["isodepth_consensus"].astype(float), o[ptcol].astype(float)
    ok = iso.notna() & pt.notna()
    gcol = grp if grp in o.columns else ("timepoint" if "timepoint" in o.columns else None)
    print(f"--- {sp} (pt col = {ptcol}) n={int(ok.sum())}")
    r_all = float(spearmanr(iso[ok], pt[ok]).statistic); print(f"    overall rho = {r_all:.4f}")
    per = {}
    if gcol:
        for g in sorted(o.loc[ok, gcol].astype(str).unique()):
            m = ok & (o[gcol].astype(str)==g)
            if m.sum() > 100:
                per[g] = round(float(spearmanr(iso[m], pt[m]).statistic),4)
                print(f"      {g:18s} n={int(m.sum()):6d}  rho={per[g]:+.4f}")
    out[sp] = (r_all, per); del a
verify("human isodepth vs pt (overall)", round(out["human"][0],3), 0.671)
verify("mouse isodepth vs pt (overall)", round(out["mouse"][0],3), 0.690)
verify("all per-group rhos positive", bool(all(v>0 for sp in out for v in out[sp][1].values())), True)

## 2.2 Cross-species transfer, per mouse timepoint — the 4dpi question

Recorded (`isodepth_pt_correlation.json`, full multi-sample object, 43,183 mouse cells):
overall 0.591; 3dpi 0.793, uninduced 0.711, 2dpi 0.702, 1dpi_late 0.596, 1dpi_early 0.583,
**4dpi 0.207**.

4dpi is degraded, *not* inverted. The negative values (−0.31 to −0.43) came only from **isolated
single-timepoint fits** and only in the secretory reference — in the full object 4dpi is positive
but weakest by ~3x.

In [ ]:
rec = {"uninduced":0.7108, "1_dpi_early":0.5829, "1_dpi_late":0.5958,
       "2_dpi":0.7023, "3_dpi":0.7926, "4_dpi_bleeding":0.2071}
p = DATA/"isodepth_pt_correlation.json"
p = p if p.exists() else None
if p:
    d = json.load(open(p))["mouse_native_iso_vs_transferred_iso"]
    print(f"overall rho = {d['spearman']:.4f} on n={d['n']:,}")
    for t, v in d["per_group"].items():
        print(f"   {t:16s} n={v['n']:6d}  rho={v['spearman']:+.4f}")
        verify(f"transfer rho {t}", round(v["spearman"],3), round(rec.get(t, np.nan),3))
    verify("overall transfer rho", round(d["spearman"],3), 0.591)
    verify("4dpi is POSITIVE (degraded, not inverted)", bool(d["per_group"]["4_dpi_bleeding"]["spearman"]>0), True)
else:
    print("isodepth_pt_correlation.json not found - recompute via the transfer job, or point p= at it")

## 2.3 Three transfer numbers that do **not** reconcile — open issue

| measurement | direction | target scalar | rho |
|---|---|---|---|
| vanilla joint object, stored column | mm->hs | human native **isodepth** | 0.389 (menstrual 0.280, secretory 0.525) |
| full-object per-timepoint (2.2 above) | hs->mm | mouse native **isodepth** | 0.591 |
| 12 per-timepoint fits | both | same scalar | 0.688 induced / 0.492 uninduced / −0.048 at 4dpi |

Different arms (vanilla vs harmony), coordinate vintages, and cell subsets. **Do not quote a single
transfer number in the manuscript until this is resolved.**

In [ ]:
a = None
try:
    import scanpy as sc
    a = sc.read_h5ad(NET/"2026_08_31_samap_vanilla_fibdec_joint.h5ad", backed="r")
    o = a.obs; sp = o["species"].astype(str)
    nv, tv = o["isodepth_consensus"].astype(float), o["isodepth_transferred"].astype(float)
    ok = (sp=="human") & nv.notna() & tv.notna()
    from scipy.stats import spearmanr
    r = float(spearmanr(nv[ok], tv[ok]).statistic)
    print(f"vanilla joint, mm->hs isodepth vs human native isodepth: rho={r:.4f} on n={int(ok.sum()):,}")
    verify("vanilla joint stored-column transfer", round(r,3), 0.389)
    print("\nNOTE: transferred columns exist on HUMAN cells only -> mouse 4dpi cannot be scored as a")
    print("      target in this object. That is why 2.2 uses the per-timepoint job instead.")
    for c in ("isodepth_transferred","palantir_pt_transferred","palantir_pt_native"):
        if c in o.columns:
            v = o[c].astype(float)
            print(f"   {c:26s} human={int((v.notna()&(sp=='human')).sum()):6d}  mouse={int((v.notna()&(sp=='mouse')).sum()):6d}")
except Exception as e:
    print("skipped:", type(e).__name__, e)
finally:
    del a

## 2.4 CAVEAT — `dpt_pseudotime` has almost no dynamic range

Discovered 2026-09-01 while auditing the 4dpi pairing. On the mouse object, diffusion pseudotime is
compressed into a sliver of [0,1] for **every** timepoint:

| timepoint | median | IQR | max | frac > 0.9 |
|---|---|---|---|---|
| uninduced | 0.395 | 0.0155 | 0.425 | 0 |
| 1_dpi_early | 0.405 | 0.0178 | 0.470 | 0 |
| 1_dpi_late | 0.410 | 0.0281 | 0.759 | 0 |
| 2_dpi | 0.422 | 0.0356 | 0.761 | 0 |
| 3_dpi | 0.436 | 0.0305 | 0.822 | 0 |
| 4_dpi_bleeding | 0.444 | 0.0413 | **1.000** | 0.0062 |

Every cell sits within ~0.05 of 0.42, with a handful of outliers reaching 1.0 (only at 4dpi).

**What this does and does not invalidate.** Spearman is invariant to monotone compression, so the
rank correlations reported above stand. But: (a) the pseudotime is not interpretable as a *magnitude*
— do not report differences or means; (b) any Pearson/linear statistic on it is dominated by the few
outliers; (c) the apparent flatness of pseudotime in the spatial staining panels is expected, not a
plotting bug; (d) it is worth checking whether DPT actually converged, since a healthy diffusion
pseudotime normally spans much of [0,1].

The medians do order correctly with time (0.395 uninduced -> 0.444 at 4dpi), so the signal is real —
just very weak in scale.

In [ ]:
import scanpy as sc
a = sc.read_h5ad(ISO/"mouse_fibdec_isodepth.h5ad", backed="r")
o = a.obs; tp = o["timepoint"].astype(str)
rows = []
for t in sorted(tp.unique()):
    v = pd.to_numeric(o.loc[tp==t, "dpt_pseudotime"], errors="coerce").dropna()
    if len(v) < 10: continue
    rows.append(dict(timepoint=t, n=len(v), median=round(float(v.median()),4),
                     iqr=round(float(v.quantile(.75)-v.quantile(.25)),4),
                     max=round(float(v.max()),4), frac_gt_0p9=round(float((v>0.9).mean()),4)))
Q = pd.DataFrame(rows)
print(Q.to_string(index=False))
verify("dpt IQR is tiny in every timepoint (<0.05)", bool((Q.iqr < 0.05).all()), True,
       note="pseudotime is rank-usable but NOT magnitude-interpretable")
verify("4dpi is the only timepoint with values > 0.9",
       bool((Q.loc[Q.frac_gt_0p9 > 0, "timepoint"] == "4_dpi_bleeding").all()), True)
# NB: Q.median would resolve to the DataFrame METHOD, not the column - always bracket-index
_order = ["uninduced","1_dpi_early","1_dpi_late","2_dpi","3_dpi","4_dpi_bleeding"]
_med = Q.set_index("timepoint").reindex(_order)["median"].dropna()
print("medians in time order:", _med.round(4).to_dict())
verify("dpt medians increase with time", bool(_med.is_monotonic_increasing), True,
       note="signal is real but the scale is tiny")
del a

## Summary

In [ ]:
check_summary()